# Quantization + Fault Injuction

the fault injection process need to quantize the model first,  inject the faults and then save it as .pt file






In [ ]:
import torch
import os
import torchvision.transforms as transforms
import torchvision.datasets as datasets
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.quantized.engine = 'qnnpack'

In [ ]:
model_path = "/content/entire_model.pth"
original_model  = torch.load(model_path, map_location=device, weights_only=False)
original_model .eval()

print("original_model  loaded successfully!")

## Quantization

you can either select a PTQ or QAT, run the needed cells and ignore the rest

### PTQ

In [ ]:
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
torch.backends.quantized.engine = 'qnnpack'  # or 'fbgemm'
qconfig = get_default_qconfig(torch.backends.quantized.engine)


In [ ]:
Calibration_transform = transforms.Compose([
    transforms.Resize (size = (224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Calibration_transform  = transforms.Compose([ # for the grayscale (Xray / MRI) datasets
#     transforms.Lambda(lambda img: img.convert("RGB")),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225]),
#                                               ])


val_dir = "LC25000 dataset/val"
# val_dir = "Chest X-Ray dataset/val"
# val_dir = "MRI dataset/val"
calibration_dataset = datasets.ImageFolder(val_dir, transform=Calibration_transform)
calibration_subset, _ = torch.utils.data.random_split(calibration_dataset, [200, len(calibration_dataset) - 200])
calibration_loader = torch.utils.data.DataLoader(calibration_subset, batch_size=32, shuffle=False)


In [ ]:
example_inputs, _ = next(iter(calibration_loader))
example_inputs = example_inputs.to("cpu")

In [ ]:
from torch.ao.quantization import QConfigMapping
# 3. FX PTQ workflow
qconfig_mapping = QConfigMapping().set_global(qconfig)
prepared_model = prepare_fx(original_model , qconfig_mapping, example_inputs)

In [ ]:
print("🔄 Running calibration ...")
prepared_model.eval()
with torch.no_grad():
    for images, _ in calibration_loader:
        images = images.to("cpu")
        prepared_model(images)
print("✅ Calibration done!")


In [ ]:
quantized_model = convert_fx(prepared_model)
quantized_model.eval()
print("✅ Model quantized successfully!")


In [ ]:
for name, param in quantized_model.named_parameters():
    print(f"{name} dtype: {param.dtype}")


In [ ]:
for name, module in quantized_model.named_modules():
    if hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
        # Regular float layer
        print(f"{name} weight dtype: {module.weight.dtype}")
    elif hasattr(module, "weight") and callable(module.weight):
        # Quantized layer
        print(f"{name} weight dtype: {module.weight().dtype}")


### QAT

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from torch.ao.quantization import (
    get_default_qat_qconfig,
    QConfigMapping
)
from torch.ao.quantization.quantize_fx import (
    prepare_qat_fx,
    convert_fx
)


from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
def replace_relu6(module):
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU6):
            setattr(module, name, nn.ReLU(inplace=True))
        else:
            replace_relu6(child)

# UNCOMMENT ONLY if your model uses ReLU6 (MNV2/GNV2)
# replace_relu6(original_model)
# print("ReLU6 replaced with ReLU ✔")


In [ ]:

# -------------------------- USE_GRAYSCALE:
train_transform = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# train_dir = "/kaggle/input/covid-pneumonia-normal-chest-xray-images/train"
# val_dir   = "/kaggle/input/covid-pneumonia-normal-chest-xray-images/val"


train_dir = "/content/drive/MyDrive/K+F MRI (clean)/train"
val_dir   = "/content/drive/MyDrive/K+F MRI (clean)/val"

In [ ]:

# train_transform = transforms.Compose([

#     transforms.Resize (size = (224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),

# ])

# val_transform = transforms.Compose([
#   transforms.Resize (size = (224, 224)),
#    transforms.ToTensor(),
#       transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),

#  ])

# train_dir = "/content/drive/MyDrive/colon_after_splitting/train"
# val_dir   = "/content/drive/MyDrive/colon_after_splitting/val"


In [ ]:

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

example_inputs, _ = next(iter(train_loader))
example_inputs = example_inputs.to("cpu")


In [ ]:
#fbgemm or qnnpack
backend = "qnnpack"
qconfig = get_default_qat_qconfig(backend)

qconfig_mapping = QConfigMapping().set_global(qconfig)
print("Model prepared for QAT ✔")


In [ ]:
original_model.to("cpu")
original_model.train()

prepared_qat_model = prepare_qat_fx(
    original_model,
    qconfig_mapping,
    example_inputs,
)
prepared_qat_model.to(device)
prepared_qat_model.train()

print("FX-QAT model prepared successfully!")

In [ ]:

pos = 1  # colon = 0, xray/MRI = 1
criterion = nn.BCEWithLogitsLoss()
optimizer_qat = torch.optim.Adam(prepared_qat_model.parameters(), lr=1e-5)

num_epochs = 7

torch.manual_seed(42)

for epoch in range(num_epochs):

    # ----------------------
    # TRAIN
    # ----------------------
    prepared_qat_model.train()
    total_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Train)", ncols=80)

    for imgs, labels in train_bar:
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        optimizer_qat.zero_grad()
        logits = prepared_qat_model(imgs).squeeze()

        loss = criterion(logits, labels)
        loss.backward()
        optimizer_qat.step()

        total_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

    # ----------------------
    # VALIDATION
    # ----------------------
    prepared_qat_model.eval()

    all_preds = []
    all_labels = []

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Val)", ncols=80)

    with torch.no_grad():
        for imgs, labels in val_bar:
            imgs = imgs.to(device)
            labels = labels.float().to(device)

            logits = prepared_qat_model(imgs).squeeze()
            preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds, pos_label = pos)
    prec = precision_score(all_labels, all_preds, zero_division=0,pos_label = pos)
    rec  = recall_score(all_labels, all_preds, zero_division=0, pos_label = pos)
    f1   = f1_score(all_labels, all_preds, zero_division=0, pos_label = pos)

    print(f"\n Epoch {epoch+1}/{num_epochs}")
    print(f"Loss:      {total_loss/len(train_loader):.4f}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")


In [ ]:
prepared_qat_model.to("cpu")
prepared_qat_model.eval()

quantized_model = convert_fx(prepared_qat_model)
print("INT8 Model Ready ")

In [ ]:
for name, m in quantized_model.named_modules():
    if hasattr(m, "weight"):
        try:
            w = m.weight()
            print(name, w.dtype)
        except:
            pass


In [ ]:
for name, module in quantized_model.named_modules():
    if hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
        # Regular float layer
        print(f"{name} weight dtype: {module.weight.dtype}")
    elif hasattr(module, "weight") and callable(module.weight):
        # Quantized layer
        print(f"{name} weight dtype: {module.weight().dtype}")


In [ ]:
for m in quantized_model.modules():
    if "Conv" in m.__class__.__name__:
        print(type(m))

## Fault Injection

In [ ]:
def print_specific_modules(model):
    print("Quantized Model (Specific Modules):")
    for name, module in model.named_children():
        if name == 'avgpool' or name == 'fc':
            print(f"  ({name}): {module}")

print_specific_modules(quantized_model)
# use this code to get the zero and scale value for injection

In [ ]:
import copy
import torch
import torch.nn as nn
from torch import fx
import math
import numpy as np
import random


fault_model = copy.deepcopy(quantized_model)

In [ ]:
seed = 0

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

num_channels = 153 #change to the vector size of NN
fault_type = "FLIP" # Options: "SA-1", "SA-0", "FLIP"
target_bits = [4,5] #(0 through 7)
percentage = 0.2

bit_val = sum(2**b for b in target_bits)

num_to_fault = math.ceil(percentage * num_channels)
target_channels = torch.randperm(num_channels)[:num_to_fault].tolist()

fault_mask = torch.zeros(1, num_channels, dtype=torch.uint8)

for ch in target_channels:
    fault_mask[0, ch] = bit_val

mode_msg = f"Bits {target_bits}"
print(f"Targeting {len(target_channels)} values: {mode_msg} set to {fault_type} (Value: {bit_val})")
print(f"Mask binary: {bin(bit_val)}")

In [ ]:
print(f"Mask binary: {bin(bit_val)}")
if fault_type == "SA-0":
    # This is what the surgery will actually apply to your data
    print(f"Effective SA-0 AND-mask: {bin(bit_val ^ 0xFF)}")

In [ ]:
# @title RN18
# ------------------ for RN18 injection
target_scale = 0.05534251034259796 # you must get the exact numbers for data integrity
target_zero_point = 0

fault_model = fx.symbolic_trace(fault_model)
fault_model.register_buffer('fault_mask', fault_mask.to(torch.uint8))
graph = fault_model.graph

get_fault_node = graph.get_attr("fault_mask")
first_node = next(iter(graph.nodes))
first_node.prepend(get_fault_node) # this is the original value

for node in graph.nodes:
    if "flatten" in node.name:
        with graph.inserting_after(node):

            int_repr_node = graph.call_method('int_repr', args=(node,))
            with graph.inserting_after(int_repr_node):
              if fault_type == "SA-1": #OR
                injection_node = graph.call_function(torch.bitwise_or,
                                                     args=(int_repr_node, get_fault_node))



              elif fault_type == "FLIP":
                injection_node = graph.call_function(torch.bitwise_xor,
                                                     args=(int_repr_node, get_fault_node))



              else: # SA-0
                    # inv_mask = graph.call_function(torch.bitwise_xor, # to flip the mask
                    #                                args=(get_fault_node,255))
                    inv_mask = graph.call_function( torch.bitwise_not,
                                                     args=(get_fault_node,))


                    inv_mask.name = "mask_flipper"

                    with graph.inserting_after(inv_mask):
                         injection_node = graph.call_function(torch.bitwise_and,
                                                         args=(int_repr_node, inv_mask))

            injection_node.name = f"bit_fault_{fault_type}"

            with graph.inserting_after(injection_node):
                requant_node = graph.call_function(
                    torch._make_per_tensor_quantized_tensor,
                    args=(injection_node, target_scale, target_zero_point)
                )

        node.replace_all_uses_with(requant_node)

        int_repr_node.update_arg(0, node)
        break

graph.lint()
fault_model.recompile()
print(f"✅ RN18")

In [ ]:
@title MNV2
# ------------------ for MNV2 injection
target_scale = 0.047535862773656845
target_zero_point = 0

fault_model = fx.symbolic_trace(fault_model)
fault_model.register_buffer('fault_mask', fault_mask.to(torch.uint8))
graph = fault_model.graph

get_fault_node = graph.get_attr("fault_mask")
first_node = next(iter(graph.nodes))
first_node.prepend(get_fault_node)

for node in graph.nodes:
    if "flatten" in node.name:  # MNV2 Boundary
        with graph.inserting_after(node):


            int_repr_node = graph.call_method('int_repr', args=(node,))
            with graph.inserting_after(int_repr_node):
              if fault_type == "SA-1":
                injection_node = graph.call_function(torch.bitwise_or,
                                                    args=(int_repr_node, get_fault_node))
              elif fault_type == "FLIP":
                injection_node = graph.call_function(torch.bitwise_xor,
                                              args=(int_repr_node, get_fault_node))


              else :# SA-0
                inv_mask = graph.call_function( torch.bitwise_not,
                                                  args=(get_fault_node,))
                inv_mask.name = "mask_flipper"


                with graph.inserting_after(inv_mask):
                  injection_node = graph.call_function(torch.bitwise_and,
                                                       args=(int_repr_node, inv_mask))

            injection_node.name = f"bit_fault_{fault_type}"

            with graph.inserting_after(injection_node):
                requant_node = graph.call_function(
                    torch._make_per_tensor_quantized_tensor,
                    args=(injection_node, target_scale, target_zero_point)
            )

        node.replace_all_uses_with(requant_node)

        int_repr_node.update_arg(0, node)
        break

graph.lint()
fault_model.recompile()
print(f"✅ MNV2")

In [ ]:
# @title RGY
# ------------------ for RGY injection
target_scale = 0.05698651075363159
target_zero_point = 0

fault_model = fx.symbolic_trace(fault_model)
fault_model.register_buffer('fault_mask', fault_mask.to(torch.uint8))
graph = fault_model.graph

get_fault_node = graph.get_attr("fault_mask")
first_node = next(iter(graph.nodes))
first_node.prepend(get_fault_node)

for node in graph.nodes:
    if node.name == "quantize_per_tensor_1":
        with graph.inserting_after(node):

            int_repr_node = graph.call_method('int_repr', args=(node,))
            with graph.inserting_after(int_repr_node):
              if fault_type == "SA-1": #OR
                injection_node = graph.call_function(torch.bitwise_or,
                                                     args=(int_repr_node, get_fault_node))



              elif fault_type == "FLIP":
                injection_node = graph.call_function(torch.bitwise_xor,
                                                     args=(int_repr_node, get_fault_node))



              else: # SA-0

                    inv_mask = graph.call_function( torch.bitwise_not,
                                                     args=(get_fault_node,))


                    inv_mask.name = "mask_flipper"

                    with graph.inserting_after(inv_mask):
                         injection_node = graph.call_function(torch.bitwise_and,
                                                         args=(int_repr_node, inv_mask))

            injection_node.name = f"bit_fault_{fault_type}"

            with graph.inserting_after(injection_node):
                requant_node = graph.call_function(
                    torch._make_per_tensor_quantized_tensor,
                    args=(injection_node, target_scale, target_zero_point)
                )

        node.replace_all_uses_with(requant_node)

        int_repr_node.update_arg(0, node)
        break

graph.lint()
fault_model.recompile()
print(f"✅ RegNetY")

In [ ]:
# Saving
example_input = torch.randn(1, 3, 224, 224)

scripted_model = torch.jit.trace(fault_model, example_input)

# Save it
script_path = "faulty_model.pt"
scripted_model.save(script_path)
print(f" {script_path}")

In [ ]:
# Define the paths to save the model
save_path_state_dict = "faulty_model.pth"
save_path_entire_model = "faulty_model.pth"

torch.save(fault_model.state_dict(), save_path_state_dict)
print(f"Model state dictionary saved to {save_path_state_dict}")
.
torch.save(fault_model, save_path_entire_model)
print(f"Entire model saved to {save_path_entire_model}")

In [ ]:
# fault_model.graph.print_tabular()

In [ ]:
# for node in fault_model.graph.nodes:
#     print(node.name)

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224)
output = fault_model(dummy_input)

print("Model Output Shape:", output.shape)
print("Raw Logic (Predictions):", output)

In [ ]:
# fault_model

# Model Evaluation

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import random
from PIL import Image
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
from pathlib import Path

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def walk_through_dir(dir_path):
    for dirpath, dirname, filenames in os.walk(dir_path):
        print(f"There are {len(dirname)} directories and {len(filenames)} images in '{dirpath}'")

dataset_path = "colon dataset"
# dataset_path = "xray dataset"
# dataset_path = "MRI dataset"
walk_through_dir(dataset_path)  #& pass the dataset path here ...

In [ ]:
test_dir  = os.path.join(dataset_path, "test")
test_dir

In [ ]:
test_transform = transforms.Compose([ # For the RGB (colon) dataset
    transforms.Resize (size = (224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),

 ])

# test_transform = transforms.Compose([ # for the grayscale (Xray / MRI) datasets
#     transforms.Lambda(lambda img: img.convert("RGB")),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225]),
# ])

In [ ]:
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
print(test_dataset)
print("-----------------------------------------------------------------------------")
print("Classes:", test_dataset.class_to_idx)
print("-----------------------------------------------------------------------------")
print("Number of test images:", len(test_dataset))


In [ ]:
# quantized_model

In [ ]:
#just like training
import torch
import torch.nn as nn
import time
from tqdm import tqdm

def evaluate_model_q(model, dataloader, device="cpu"):
    model.eval()
    model.to(device)  # quantized models usually on CPU

    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels.float().unsqueeze(1))
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).int() #long(),using > 0.5 just like in the validation

            correct += (predicted.view(-1) == labels.int().view(-1)).sum().item() # view(-1) make sure theres no shape mismatch
            total += labels.numel() # or labels.size(0)

    end_time = time.time()
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)

    print(f"\n✅ Accuracy: {accuracy:.2f}%")
    print(f"📉 Average Loss: {avg_loss:.4f}")
    print(f"⏱️ Evaluation Time: {end_time - start_time:.2f}s")

    return accuracy, avg_loss


In [ ]:
print ("Quantized model testing:")
evaluate_model_q(quantized_model, test_loader, device="cpu")

In [ ]:
print ("fault model testing:")
evaluate_model_q(fault_model, test_loader, device="cpu")

## Methodology: Quantized Model Fault Injection

This section details the methodology employed for injecting faults into quantized neural network models, specifically targeting the computational graph to simulate hardware-level errors. The approach leverages PyTorch's FX tracing capabilities to precisely insert fault operations at critical points within the model's inference pipeline.

### 1. Model Preparation and Quantization

Prior to fault injection, the target deep learning model undergoes quantization. This involves converting floating-point (FP32) weights and activations into lower-bit integer representations (typically INT8). The quantization process in this study follows a Post-Training Quantization (PTQ) workflow using `torch.ao.quantization.quantize_fx`. A calibration step is performed using a subset of the validation dataset to determine optimal quantization parameters (scales and zero-points) for the activations. The resultant `quantized_model` is an FX-traced graph where operations are represented by their quantized equivalents.

### 2. Fault Injection Strategy

The primary objective of the fault injection is to introduce bit-level perturbations directly into the integer representation of activation values. This is achieved by transforming the model's computational graph to include fault injection nodes. The injection point is strategically chosen at the boundary between the Convolutional Neural Network (CNN) feature extractor and the Artificial Neural Network (ANN) classifier, specifically after the `flatten` layer. This location is critical as it represents the consolidated feature vector passed to the decision-making layers, making it a sensitive point for error propagation.

#### 2.1. Bit-level Fault Mechanism

Faults are applied to the 8-bit unsigned integer (`uint8`) representation of the quantized activations. The process is as follows:

1.  **Fault Parameter Definition**: Key parameters for fault injection are defined:
    *   `seed`: For reproducibility of random channel selection.
    *   `num_channels`: The total number of channels in the activation vector after the `flatten` layer (e.g., 512 for ResNet-18). These correspond to the number of individual values in the 1D feature vector.
    *   `fault_type`: Specifies the type of bit-level fault: "SA-0" (stuck-at 0), "SA-1" (stuck-at 1), or "FLIP" (bit-flip).
    *   `target_bits`: A list of bit positions (0-7, where 0 is the Least Significant Bit) within the 8-bit representation to be affected.
    *   `percentage`: The proportion of `num_channels` to be targeted for fault injection. This determines `num_to_fault` and the `target_channels` are randomly selected from the `num_channels` based on this percentage.

2.  **Fault Mask Generation**: A `fault_mask` is created as a `torch.uint8` tensor with a shape of `(1, num_channels)`. For each randomly selected `target_channel`, a `bit_val` is computed by summing `2**b` for all `b` in `target_bits`. This `bit_val` is then placed in the `fault_mask` at the corresponding channel position.

3.  **FX Graph Transformation**: `torch.fx.symbolic_trace` is utilized to obtain a symbolic representation of the `quantized_model`. The `fault_mask` is registered as a buffer (`fault_model.register_buffer('fault_mask', ...)`), making it accessible within the traced graph.

4.  **Insertion of Fault Nodes**: The computational graph is modified to insert custom operations after the `flatten` layer:
    *   **`int_repr_node`**: The quantized tensor output from the `flatten` layer is converted to its raw `uint8` integer representation using `torch.Tensor.int_repr()`.
    *   **`injection_node`**: Based on the `fault_type`:
        *   **SA-1**: A bitwise OR operation (`torch.bitwise_or`) is performed between the `int_repr_node` and the `fault_mask`.
        *   **FLIP**: A bitwise XOR operation (`torch.bitwise_xor`) is performed between the `int_repr_node` and the `fault_mask`.
        *   **SA-0**: A bitwise AND operation (`torch.bitwise_and`) is performed. To achieve SA-0, an inverse mask is first generated by performing `torch.bitwise_not` on the `fault_mask`. This inverse mask effectively sets the target bits to 0 while leaving others unchanged.
    *   **`requant_node`**: The modified integer values from the `injection_node` are re-quantized back into a `torch.qint8` tensor using `torch._make_per_tensor_quantized_tensor`. This step is crucial to ensure that the fault-injected activations retain their original scale and zero-point parameters, thereby ensuring quantization integrity, preserving the original value mapping, and maintaining compatibility by keeping the vector in a valid format for subsequent quantized layers to process them correctly.

5.  **Graph Recompilation**: The original node representing the output of the `flatten` layer is replaced with the `requant_node`, effectively integrating the fault injection logic into the model's forward pass. The modified graph is then linted and recompiled (`fault_model.recompile()`).

### 3. Evaluation of Fault-Injected Models

The `fault_model` is evaluated on a dedicated test dataset. The performance metrics, including accuracy, precision, recall, and F1-score, are then compared against the baseline `quantized_model` to quantify the impact of the injected faults on model reliability and robustness.

#### Idea Behind Fault Injection Code

Your fault injection code utilizes PyTorch's FX tracing capabilities to surgically inject faults directly into the computational graph of your quantized model. This allows for realistic simulation of hardware-level errors at specific points in the model's inference pipeline.

There are two main types of fault injection implemented:

1.  **Bit-level Fault Injection:**
    *   **Mechanism:** This method targets the **8-bit integer representation** of activation values after the `flatten` layer, right before the fully connected (ANN) part of the network.
    *   **Process:** It identifies specific bits within selected output channels and applies bitwise operations (`AND`, `OR`, `XOR`) to simulate `SA-0` (stuck-at 0), `SA-1` (stuck-at 1), or `FLIP` (bit-flip) faults.
    *   **Re-quantization:** After bit manipulation, the modified integer values are converted back into quantized tensors, preserving the correct scale and zero-point for subsequent quantized layers.

2.  **Value-based Fault Injection:**
    *   **Mechanism:** This approach injects faults into the **floating-point representation** of activation values after dequantization, also at the CNN-ANN boundary.
    *   **Process:** It defines a `fault_mask` (e.g., zeroing out specific channels for `SA-0` or `SA-1`, or adding a fixed offset) and applies this mask directly to the dequantized activations.
    *   **Re-quantization:** The perturbed floating-point values are then re-quantized back to INT8 before being fed to the next quantized layers.

Both methods leverage `torch.fx.symbolic_trace` to transform the model into a graph, enabling the insertion of these fault injection operations as permanent parts of the model's forward pass. This is a robust and flexible way to evaluate the fault tolerance of the quantized neural networks.

# Save

In [ ]:
scripted_model = torch.jit.script(quantized_model)
scripted_model.save("quantized_jit_fauty_model.pt")


In [ ]:
save_path_state_dict = "quantized_state_fauty_model.pth"
save_path_entire_model = "quantized_entire_fauty_model.pth"

torch.save(quantized_model.state_dict(), save_path_state_dict)
print(f"Model state dictionary saved to {save_path_state_dict}")

torch.save(quantized_model, save_path_entire_model)
print(f"Entire model saved to {save_path_entire_model}")

In [ ]:
model_path = "/content/quantized_jit_faulty_model.pt"
Q_model_pt  = torch.jit.load(model_path, map_location=device)
Q_model_pt.eval()

print("✅")